In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import fnmatch
import matplotlib.pyplot as plt
from functools import reduce
from bs4 import BeautifulSoup
import requests
import osmnx as ox
from API_KEY import get_OneMap_token
import networkx as nx
import copy
import matplotlib
import matplotlib as mpl

import importlib

import helper_functions.utils as utils
import helper_functions.serviceArea
import helper_functions.amenities_dict
import helper_functions.OneMapAPI
importlib.reload(helper_functions.serviceArea)
importlib.reload(helper_functions.amenities_dict)
importlib.reload(helper_functions.OneMapAPI)
import helper_functions.serviceArea as serviceArea
import helper_functions.amenities_dict as amenities_dict
import helper_functions.plot_utils as plot_utils
import helper_functions.OneMapAPI as OneMapAPI

["Since 1969, the HDB has made a conscious decision to keep the ground floor of blocks free of housing units and only allow communal amenities such as kindergartens, childcare centres and senior citizens clubs."](https://www.nlb.gov.sg/main/article-detail?cmsuuid=e342b869-736c-4e4c-a346-47b164663572)

# Import dataset

In [23]:
rental_prices_df = pd.read_csv(r"Exported_Data\Rental_prices_2014_2024.csv")
print(rental_prices_df.dtypes)
rental_prices_df["Lease Commencement Format Date"] = pd.to_datetime(rental_prices_df["Lease Commencement Date"],format="%b %Y")
rental_prices_df.head()

C:\Users\hypak\AppData\Local\Temp\ipykernel_26016\1760203481.py:1: DtypeWarning: Columns (12,16) have mixed types. Specify dtype option on import or set low_memory=False.
  rental_prices_df = pd.read_csv(r"Exported_Data\Rental_prices_2014_2024.csv")


Project Name                object
Street Name                 object
Postal District              int64
Property Type               object
No of Bedroom              float64
Monthly Rent ($)            object
Floor Area (SQM)            object
Floor Area (SQFT)           object
Lease Commencement Date     object
Area (SQM)                 float64
Area (SQFT)                float64
SEARCHVAL                   object
BLK_NO                      object
ROAD_NAME                   object
BUILDING                    object
ADDRESS                     object
POSTAL                      object
X                          float64
Y                          float64
LATITUDE                   float64
LONGITUDE                  float64
dtype: object


,Project Name,Street Name,Postal District,Property Type,No of Bedroom,Monthly Rent ($),Floor Area (SQM),Floor Area (SQFT),Lease Commencement Date,Area (SQM),...,BLK_NO,ROAD_NAME,BUILDING,ADDRESS,POSTAL,X,Y,LATITUDE,LONGITUDE,Lease Commencement Format Date
0,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,2.0,"2,300",50 to 60,500 to 600,Nov 2019,50.0,...,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.161877,37007.879327,1.350961,103.835504,2019-11-01
1,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,3.0,"3,300",70 to 80,700 to 800,Jan 2020,70.0,...,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.161877,37007.879327,1.350961,103.835504,2020-01-01
2,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,3.0,"4,000",100 to 110,"1,100 to 1,200",Apr 2020,100.0,...,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.161877,37007.879327,1.350961,103.835504,2020-04-01
3,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,2.0,"2,500",50 to 60,500 to 600,Jan 2020,50.0,...,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.161877,37007.879327,1.350961,103.835504,2020-01-01
4,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,2.0,"2,300",50 to 60,500 to 600,Dec 2019,50.0,...,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.161877,37007.879327,1.350961,103.835504,2019-12-01


# Data cleaning of rental prices

In [2]:
rental_prices_dir = r"Data\Rental Search"
rental_prices_fps = [os.path.join(rental_prices_dir,d,fp) for d in os.listdir(rental_prices_dir) for fp in os.listdir(os.path.join(rental_prices_dir,d))]

In [3]:
rental_prices_dfs = []
for fp in rental_prices_fps:
    try:
        df = pd.read_csv(fp)
    except Exception as e:
        df = pd.read_csv(fp, encoding='latin-1')
    rental_prices_dfs.append(df)
rental_prices_dfs[0]

rental_prices_df = pd.concat(rental_prices_dfs)
print(f"length of df: {len(rental_prices_df)}")
rental_prices_df.head()

length of df: 978567


,Project Name,Street Name,Postal District,Property Type,No of Bedroom,Monthly Rent ($),Floor Area (SQM),Floor Area (SQFT),Lease Commencement Date
0,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,2.0,"2,300",50 to 60,500 to 600,Nov 2019
1,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,3.0,"3,300",70 to 80,700 to 800,Jan 2020
2,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,3.0,"4,000",100 to 110,"1,100 to 1,200",Apr 2020
3,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,2.0,"2,500",50 to 60,500 to 600,Jan 2020
4,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,2.0,"2,300",50 to 60,500 to 600,Dec 2019


In [4]:
# rental_prices_df["Area (SQM)"] = 
rental_prices_df["Area (SQM)"] = rental_prices_df["Floor Area (SQM)"].str.split(' to ',expand=True)[0].str.replace(',', '').astype(float)
rental_prices_df["Area (SQFT)"] = rental_prices_df["Floor Area (SQFT)"].str.split(' to ',expand=True)[0].str.replace(',', '').astype(float)

In [13]:
headers = OneMapAPI.generate_OneMap_headers()
response = OneMapAPI.get_coordinates_from_location(rental_prices_df["Project Name"].values[0],headers)
new_headers = list(response)
new_headers

def postal_to_coord(row,column_names,headers):
    try:
        searchVal = OneMapAPI.get_coordinates_from_location(row,headers)
    except:
        searchVal = {c:np.nan for c in column_names}
    return pd.Series(searchVal,index=list(searchVal))

rental_proj_name_searchVal = pd.Series(rental_prices_df["Project Name"].unique()).apply(lambda x: postal_to_coord(x, new_headers,headers))
rental_proj_name_searchVal.head()
# merge search val with rental_prices_df based on search val and project name

,SEARCHVAL,BLK_NO,ROAD_NAME,BUILDING,ADDRESS,POSTAL,X,Y,LATITUDE,LONGITUDE
0,183 LONGHAUS,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.1618774943,37007.8793265987,1.35096063593161,103.835503542512
1,3BHC,3A,BRIGHT HILL CRESCENT,3BHC,3A BRIGHT HILL CRESCENT 3BHC SINGAPORE 574003,574003,27924.9522340391,37600.8611208797,1.35632335310832,103.832644225665
2,ADANA @ THOMSON,10,OLD UPPER THOMSON ROAD,ADANA @ THOMSON,10 OLD UPPER THOMSON ROAD ADANA @ THOMSON SING...,573869,27392.7925001755,39059.7647997476,1.36951715154198,103.827862392894
3,ADELPHI PARK ESTATE,97,GARDENIA ROAD,ADELPHI PARK ESTATE,97 GARDENIA ROAD ADELPHI PARK ESTATE SINGAPORE...,578870,27304.0559128476,37370.7000129484,1.35424184993566,103.82706507179
4,ALANA,202,SUNRISE TERRACE,ALANA,202 SUNRISE TERRACE ALANA SINGAPORE 804578,804578,30414.6770160878,41186.0968910921,1.38874684430529,103.85501632152


In [ ]:
rental_prices_df = pd.merge(rental_prices_df,rental_proj_name_searchVal,how="inner",
         right_on="SEARCHVAL",left_on="Project Name")

rental_prices_df.head()

,Project Name,Street Name,Postal District,Property Type,No of Bedroom,Monthly Rent ($),Floor Area (SQM),Floor Area (SQFT),Lease Commencement Date,Area (SQM),...,SEARCHVAL,BLK_NO,ROAD_NAME,BUILDING,ADDRESS,POSTAL,X,Y,LATITUDE,LONGITUDE
0,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,2.0,"2,300",50 to 60,500 to 600,Nov 2019,50.0,...,183 LONGHAUS,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.1618774943,37007.8793265987,1.35096063593161,103.835503542512
1,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,3.0,"3,300",70 to 80,700 to 800,Jan 2020,70.0,...,183 LONGHAUS,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.1618774943,37007.8793265987,1.35096063593161,103.835503542512
2,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,3.0,"4,000",100 to 110,"1,100 to 1,200",Apr 2020,100.0,...,183 LONGHAUS,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.1618774943,37007.8793265987,1.35096063593161,103.835503542512
3,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,2.0,"2,500",50 to 60,500 to 600,Jan 2020,50.0,...,183 LONGHAUS,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.1618774943,37007.8793265987,1.35096063593161,103.835503542512
4,183 LONGHAUS,UPPER THOMSON ROAD,20,Non-landed Properties,2.0,"2,300",50 to 60,500 to 600,Dec 2019,50.0,...,183 LONGHAUS,183,UPPER THOMSON ROAD,183 LONGHAUS,183 UPPER THOMSON ROAD 183 LONGHAUS SINGAPORE ...,574429,28243.1618774943,37007.8793265987,1.35096063593161,103.835503542512


In [18]:
print(f"length of df: {len(rental_prices_df)}")

length of df: 890511


In [ ]:
save_dir = os.path.join(os.getcwd(),"Exported_Data")
if not os.path.exists(save_dir):
    os.mkdir(save_dir)
# rental_prices_df.to_csv(os.path.join(save_dir,"Rental_prices_2014_2024.csv"),index=False)